# Show where the Distinct-G1 model may have gaps

June 13, 2024

**Goal:** Show that the distinct-G1 model may have gaps by showing that perhaps there is noise in identifiability introduced in DG1 phase genes. 

**Notes:**
- How many CG1 genes are there?
- How many DG1 genes are there?
- How does this track with the chromatin?


In [56]:
# Preamble, notebook setup and imports

%load_ext autoreload
%autoreload 2
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.figure_configs import save_figure_for_paper, FiguresConfig


In [2]:
# In Guo, 2011, DG1 genes were computed as genes with 1.5x expression in DG1 compared to CG1
# Let's see if we can reproduce the methodology
from src.gene_expression_deconv_analysis import GeneExpressionAnalysis

ge_analysis = GeneExpressionAnalysis('output/deconvolve_combined_g_opt_2024_03_14/gene_expression/')
ge_analysis.load_gene_expression_fs()


In [5]:
from src.config import load_configs_by_config_type

config1, config2 = load_configs_by_config_type('distinct')

In [29]:
ge_analysis.gene_expression_f

d_indices = config1.get_Hpositions_for_phase('DG1')
c_indices = config1.get_Hpositions_for_phase('CG1')

max_dg1 = ge_analysis.gene_expression_f[d_indices].max(axis=1)
max_cg1 = ge_analysis.gene_expression_f[c_indices].max(axis=1)

# Criteria: maximal mother/daughter expression must be 1.5x daughter/mother
dg1_genes = max_dg1.index[max_dg1 > 1.5 * max_cg1]
cg1_genes = max_dg1.index[max_cg1 > 1.5 * max_dg1]

print(f"From the Guo, 2011 methodology there are {len(dg1_genes)} daughter expression genes"
      f"\n    and {len(cg1_genes)} mother expression genes in our model results")


From the Guo, 2011 methodology there are 15 daughter expression genes
    and 1 mother expression genes in our model results


In [32]:
from src.geneset import get_deconvolved_geneset
genes = get_deconvolved_geneset()

In [68]:
genes.loc[dg1_genes].head()

,gene,chr,cat,start,stop,strand,classification,length,TSS,PAS,manually_curated,promoter_start,promoter_end,gene_body_start,gene_body_end
orf_name,,,,,,,,,,,,,,,
YDL227C,HO,4,gene,46271,48031,-,Verified,1760,48071,46212.0,False,48071.0,48371.0,47571.0,48071.0
YDL179W,PCL9,4,gene,138291,139205,+,Verified,914,138197,139358.0,False,137897.0,138197.0,138197.0,138697.0
YER124C,DSE1,5,gene,407342,409063,-,Verified,1721,409104,407002.0,False,409104.0,409404.0,408604.0,409104.0
YGR044C,RME1,7,gene,582990,583892,-,Verified,902,583968,582726.0,False,583968.0,584268.0,583468.0,583968.0
YGR109C,CLB6,7,gene,705359,706501,-,Verified,1142,706581,705248.0,False,706581.0,706881.0,706081.0,706581.0


In [34]:
genes.loc[cg1_genes]

,gene,chr,cat,start,stop,strand,classification,length,TSS,PAS,manually_curated,promoter_start,promoter_end,gene_body_start,gene_body_end
orf_name,,,,,,,,,,,,,,,
YAL067C,SEO1,1,gene,7235,9016,-,Verified,1781,9016,NaN,NaN,9016.0,9316.0,8516.0,9016.0


In [36]:
from src.promoter_ptr_analysis import PromoterPTRAnalysis

chromatin_dir = 'output/deconvolve_combined_g_opt_2024_03_14/chromatin/'
promoter_analysis = PromoterPTRAnalysis(chromatin_dir)
promoter_analysis.load_f_files()

Shape of the loaded F: (227, 200)
1/5574 - 00:00:00.150
1001/5574 - 00:00:06.168
2001/5574 - 00:00:11.226
3001/5574 - 00:00:16.021
4001/5574 - 00:00:20.764
5001/5574 - 00:00:25.519


In [52]:
# Compute the mean across the bins
all_gene_fs = promoter_analysis.all_gene_fs_df.values
all_gene_fs = all_gene_fs.reshape((-1, 227, 200))
all_gene_fs_mean = all_gene_fs.mean(axis=2)


In [59]:
chromatin_f_means = pd.DataFrame(all_gene_fs_mean, index=promoter_analysis.all_gene_fs_df.index)

In [63]:
d_max = chromatin_f_means[d_indices].max(axis=1)
c_max = chromatin_f_means[c_indices].max(axis=1)

In [72]:
ms_genes = d_max.index[c_max > 1.5 * d_max]
ds_genes = d_max.index[d_max > 1.5 * c_max]

print(f"From the Guo, 2011 methodology there are {len(ds_genes)} daughter chromatin genes"
      f"\n    and {len(ms_genes)} mother maximal chromatin genes in our model results")


From the Guo, 2011 methodology there are 548 daughter chromatin genes
    and 0 mother maximal chromatin genes in our model results
